<a href="https://colab.research.google.com/github/DhimanTarafdar/AAA/blob/main/Improve_Plant_Disease_Detection_using_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install & Import Libraries**

In [2]:
# Install kagglehub if not already installed
!pip install kagglehub -q

In [1]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import kagglehub

# **Device Setup & Dataset Download**

In [2]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

Using Device: cuda


In [3]:
# Download PlantVillage dataset
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("Dataset Path:", path)

TRAIN_PATH = os.path.join(path, "PlantVillage", "train")
VAL_PATH   = os.path.join(path, "PlantVillage", "val")

classes = os.listdir(TRAIN_PATH)
num_classes = len(classes)
print(f"Total Classes: {num_classes}")

Using Colab cache for faster access to the 'plantvillage' dataset.
Dataset Path: /kaggle/input/plantvillage
Total Classes: 38


# **Data Augmentation & Loading**

In [4]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

# Training transforms — more aggressive augmentation for real-world robustness
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMAGE_SIZE),            # Random crop instead of fixed resize
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),                # Increased rotation range
    transforms.ColorJitter(brightness=0.3,        # Handle lighting variation
                           contrast=0.3,
                           saturation=0.3,
                           hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])   # ImageNet mean/std (required for VGG16)
])

# Validation transforms — no augmentation, just resize
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [5]:
train_dataset = datasets.ImageFolder(root=TRAIN_PATH, transform=train_transform)
val_dataset   = datasets.ImageFolder(root=VAL_PATH,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Training Images  : {len(train_dataset)}")
print(f"Validation Images: {len(val_dataset)}")

Training Images  : 43444
Validation Images: 10861
